<a href="https://colab.research.google.com/github/Santiago-Echeverri-Arteaga/Fisica_Computacional_2/blob/master/curso_2026_2/04_vision/41_lenet_alexnet.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg"
       alt="Abrir en Colab"/>
</a>

# LeNet-5 y AlexNet

**Pregunta guía:** ¿Qué cambió al pasar de dígitos pequeños a visión profunda?<br>
**Duración sugerida:** 4 horas.<br>
**Entorno:** CPU; datos incluidos o generados en memoria.

El orden de trabajo es siempre: problema → matemática → implementación
mínima → biblioteca → evaluación → interpretación física.


**Requiere PyTorch.** LeNet-5 fijó el patrón convolución–submuestreo–capa
densa. AlexNet escaló profundidad, datos y cómputo, usando ReLU, GPU,
augmentación y dropout. Entrenaremos una adaptación de LeNet sobre los
dígitos 8×8 incluidos en scikit-learn, reescalados a 32×32. AlexNet se
inspecciona sin entrenarlo para no convertir la clase en una espera.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.datasets import load_digits
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision.models import alexnet

SEMILLA = 42
torch.manual_seed(SEMILLA)
dispositivo = torch.device("cuda" if torch.cuda.is_available() else "cpu")
datos = load_digits()
X = torch.tensor(datos.images[:,None]/16.0, dtype=torch.float32)
X = F.interpolate(X, size=(32,32), mode="bilinear", align_corners=False)
y = torch.tensor(datos.target, dtype=torch.long)
índices = np.arange(len(y))
dev, test = train_test_split(índices, test_size=.2, stratify=y, random_state=SEMILLA)
train, val = train_test_split(dev, test_size=.2, stratify=y[dev], random_state=SEMILLA)
loader_train = DataLoader(TensorDataset(X[train],y[train]), batch_size=64, shuffle=True, generator=torch.Generator().manual_seed(SEMILLA))
loader_val = DataLoader(TensorDataset(X[val],y[val]), batch_size=256)


In [ ]:
class LeNet5(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 6, kernel_size=5), nn.Tanh(), nn.AvgPool2d(2),
            nn.Conv2d(6, 16, kernel_size=5), nn.Tanh(), nn.AvgPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(16*5*5,120), nn.Tanh(),
            nn.Linear(120,84), nn.Tanh(), nn.Linear(84,10),
        )
    def forward(self,x): return self.classifier(self.features(x))

modelo = LeNet5().to(dispositivo)
criterio = nn.CrossEntropyLoss()
optimizador = torch.optim.Adam(modelo.parameters(), lr=1e-3)
for época in range(12):
    modelo.train()
    for xb,yb in loader_train:
        xb,yb = xb.to(dispositivo),yb.to(dispositivo)
        optimizador.zero_grad(); pérdida=criterio(modelo(xb),yb)
        pérdida.backward(); optimizador.step()
    modelo.eval()
    with torch.no_grad():
        aciertos=sum((modelo(xb.to(dispositivo)).argmax(1).cpu()==yb).sum().item() for xb,yb in loader_val)
    if época in [0,3,7,11]: print(época+1, aciertos/len(val))


In [ ]:
modelo.eval()
with torch.no_grad(): pred=modelo(X[test].to(dispositivo)).argmax(1).cpu().numpy()
print("accuracy test:", accuracy_score(y[test],pred))
ConfusionMatrixDisplay.from_predictions(y[test],pred,cmap="Blues")
plt.show()

alex = alexnet(weights=None)
print("Parámetros LeNet:", sum(p.numel() for p in modelo.parameters()))
print("Parámetros AlexNet:", sum(p.numel() for p in alex.parameters()))
print("Salida AlexNet para 2 imágenes RGB:", alex(torch.randn(2,3,224,224)).shape)
del alex


**Ejercicios:** cambie tanh/average pooling por ReLU/max pooling; haga una
ablación justa; calcule manualmente cada tamaño; explique por qué ImageNet
y disponibilidad de GPU fueron tan importantes como la arquitectura.
